# Why households stall: recorded barriers, setup friction, and what ownership costs

**Claim cluster:** `C082`, `C087`, `C088`, `C113`, `C114`  
**Source of truth:** `data/smarthome.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in these notebooks reads only from data/smarthome.duckdb.
CLAIM_IDS = ('C082', 'C087', 'C088', 'C113', 'C114')
assert viz.DEFAULT_DB_PATH.endswith("smarthome.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

The claims in this cluster are where the corpus does its own summarising: `C082` is an inference that device counts overstate engagement by 2-3x, `C087` is an inference that the post-purchase gap is a product-design problem, `C088` is a documented fact warning that the business analysis cannot yet support company-level conclusions, and `C113` is a reported insurer-pilot figure. The panels below keep the survey numbers and the corpus's own judgements apart.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### Barriers, each against the population it was measured in

The corpus records barrier rates against different populations and in different countries: adopters and non-adopters (`M025`, 46 vs 52), a general respondent pool (`M026`, 42), a privacy-and-security pair stated together (`M027`, about 45 each), a Spanish sample (`M028`, 53) and a French sample (`M029`, 82 price and 78 hacking). Each bar carries the denominator from its own `unit` cell in the hover; the bars are not averaged, because there is no single population behind them.

In [ ]:
# (metric id, bar label, index of the number to use). Some value cells state two
# populations in one string ("46 vs 52"); parse_numbers keeps the order the cell
# writes them, and the label says which population that position is.
BARRIER_ROWS = [
    ("M025", "Cost barrier, adopters", 0),
    ("M025", "Cost barrier, non-adopters", 1),
    ("M026", "Cost makes connected products less appealing", 0),
    ("M027", "Privacy and security concerns (stated as 'about 45% each')", 0),
    ("M028", "Cost barrier, Spain", 0),
    ("M029", "Price concern, France", 0),
    ("M029", "Hacking concern, France", 1),
]
_barrier_metrics = viz.metrics_by_id(["M025", "M026", "M027", "M028", "M029"])
_barrier_metrics = _barrier_metrics.set_index("local_id")
_rows = []
for _mid, _label, _index in BARRIER_ROWS:
    if _mid not in _barrier_metrics.index:
        continue
    _record = _barrier_metrics.loc[_mid]
    _numbers = viz.parse_numbers(_record["value"])
    if _index >= len(_numbers):
        continue
    _rows.append({"metric": _mid, "barrier": _label, "pct": _numbers[_index],
                  "stated": _record["value"], "unit": _record["unit"],
                  "confidence": _record["confidence"],
                  "source": _record["source_ref"]})
barriers = pd.DataFrame(_rows).sort_values("pct", ascending=False)
show(barriers)

fig = go.Figure(go.Bar(
    x=barriers["pct"],
    y=barriers["barrier"] + " (" + barriers["metric"] + ")",
    orientation="h",
    marker_color=[viz.CONFIDENCE_COLORS.get(c, viz.CONFIDENCE_COLORS[None])
                  for c in barriers["confidence"]],
    text=["%g%%" % value for value in barriers["pct"]],
    textposition="outside",
    customdata=list(zip(barriers["stated"], barriers["unit"],
                        barriers["source"], barriers["confidence"])),
    hovertemplate=("%{y}<br>stated: %{customdata[0]} (unit: %{customdata[1]})"
                   "<br>source: %{customdata[2]}"
                   "<br>confidence: %{customdata[3]}<extra></extra>"),
))
fig.update_layout(
    title=("Recorded adoption barriers, each against its own stated "
           "population (not one shared denominator)"),
    xaxis_title="percent, as stated",
    height=420,
    yaxis=dict(automargin=True, autorange="reversed"),
)
fig.show()

## Supporting context

### What ownership costs, and what setup costs in frustration

Three recorded money figures describe the same household on three different clocks: `M036` is one-off upfront spend (3,750 USD), `M037` is monthly ongoing spend (70 USD per month) and `M035` is annual frustration spending (340 USD). They are drawn with their clocks on the labels rather than added together. The setup rows (`M019`-`M024`) are a different unit family again: rates, a count of usability issues, Net Promoter Scores and a repair cost.

In [ ]:
display(HTML(
    "<h4>Recorded household money, each on its own clock</h4>"
    "<p>One-off, monthly and annual figures are not summed: the corpus "
    "records no conversion between them and no household profile to convert "
    "with.</p>"
))
_money = viz.metrics_by_id(["M036", "M037", "M035"])
show(_money[["local_id", "metric_name", "value", "unit", "confidence",
             "conditions_stated", "source_ref"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "conditions_stated": "cond_stated", "source_ref": "source",
             "confidence": "conf"}))

fig = go.Figure(go.Bar(
    x=[viz.numeric_value(value) for value in _money["value"]],
    y=[f"{record['local_id']} {record['metric_name']} ({record['unit']})"
       for record in _money.to_dict("records")],
    orientation="h",
    marker_color=[viz.CONFIDENCE_COLORS.get(c, viz.CONFIDENCE_COLORS[None])
                  for c in _money["confidence"]],
    text=["%g" % viz.numeric_value(value) for value in _money["value"]],
    textposition="outside",
    customdata=list(zip(_money["value"], _money["unit"],
                        _money["conditions_stated"])),
    hovertemplate=("%{y}<br>stated: %{customdata[0]} %{customdata[1]}"
                   "<br>conditions stated: %{customdata[2]}<extra></extra>"),
))
fig.update_layout(
    title=("Recorded household spend by clock (one-off, per month, per year): "
           "three different quantities, not a total"),
    xaxis_title="value, in the unit named on the bar",
    height=320,
    yaxis=dict(automargin=True, autorange="reversed"),
)
fig.show()

display(HTML("<h4>Setup and connectivity friction, as recorded</h4>"))
show(viz.metrics_by_id(["M019", "M020", "M021", "M022", "M023", "M024",
                        "M030"])[
    ["local_id", "metric_name", "value", "unit", "confidence", "source_ref"]]
     .rename(columns={"local_id": "metric", "metric_name": "name",
                      "source_ref": "source", "confidence": "conf"}))

display(HTML(
    "<h4>Market sizes, with currencies as stated</h4>"
    "<p>Five market and revenue figures are recorded with different units and "
    "two currencies. They are not converted or summed.</p>"
))
show(viz.metrics_by_id(["M047", "M049", "M050", "M051", "M052"])[
    ["local_id", "metric_name", "value", "unit", "confidence",
     "conditions_stated", "source_ref"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "conditions_stated": "cond_stated", "source_ref": "source",
             "confidence": "conf"}))
display(HTML(
    "<h4>Claim cluster evidence table</h4>"
    "<p>Every claim in this cluster, with its evidence class, confidence, "
    "workflow stage and cited source ids. This is the table the figures above "
    "are built from.</p>"
))
show(
    claims[["local_id", "claim_type", "evidence_marker", "confidence",
            "workflow_stage", "source_refs", "falsifier_missing", "section"]]
    .rename(columns={
        "local_id": "claim", "claim_type": "type", "evidence_marker": "ev",
        "confidence": "conf", "workflow_stage": "stage",
        "source_refs": "sources", "falsifier_missing": "no_falsifier",
    })
)

display(HTML("<h4>Sources cited by the cluster</h4>"))
show(sources[["local_id", "title", "publisher", "classification",
              "publication_date", "url"]].rename(
    columns={"local_id": "source"}))

## Uncertainty and gaps

### Six metrics in the corpus have no stated conditions

The extraction log records `M017`, `M034`, `M035`, `M038`, `M049` and `M104` as carrying the placeholder 'conditions not stated in source'. Two of them (`M035` frustration spending, `M049` market size) appear on this page, so the list is printed in full below rather than only where it is convenient.

Other limits on this page:

- **Barrier rates are not one measurement.** A share of adopters, a share of non-adopters, a share of Spanish respondents and a share of French respondents cannot be averaged into one barrier index, so the figure keeps them separate.
- **`C113` is a single pilot figure.** The insurer paying 47 EUR per policy comes from one Samsung/HSB pilot, recorded as reported signal at low confidence, and is not a market price.
- **`C088` warns against the conclusion this page invites.** The corpus states its business analysis cannot yet support company-level investment conclusions without audited metrics and comparable definitions of active users, churn, service revenue and support cost.
- **`C082` and `C087` are inferences.** 'Device counts overstate engagement by 2-3x' and 'the post-purchase gap is a product-design problem' are the corpus's readings, not measurements.
- **`M027` states two concerns in one cell.** Privacy and security concerns are recorded as 'about 45% each', so they share one bar here rather than being split into two bars the corpus did not separate.

In [ ]:
# The full unstated-conditions list comes from the schema's own view, so this
# page does not depend on which metrics happen to link to this cluster.
_con = viz.connect_db()
try:
    _unstated = pd.DataFrame(
        _con.execute("SELECT local_id, metric_name FROM unstated_conditions "
                     "ORDER BY local_id").fetchall(),
        columns=["metric", "name"],
    )
    _unstated_total = _con.execute(
        "SELECT COUNT(1) FROM unstated_conditions").fetchone()[0]
finally:
    _con.close()
display(HTML(
    f"<h4>Every metric the corpus records without conditions "
    f"({_unstated_total} in total)</h4>"
    "<p>Source: the <code>unstated_conditions</code> view, which reads "
    "<code>conditions_stated = false</code>.</p>"
))
show(_unstated)
display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

Barrier, money and market rows are read by explicit metric id with `viz.metrics_by_id` and parsed with `viz.parse_numbers`, so a value cell holding two populations ('46 vs 52') is split by position with the label stating which position is which. The `unstated_conditions` view is queried directly for the gap list.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()